# Cost Optimization for AI Agents: Batching
**Example context: a Data Analysis Agent**

Strategy covered: **Batching — process bulk work at half price.**

| Fact | Value |
|---|---|
| Batch discount | 50% on input *and* output tokens |
| Stacks with caching | cached input inside a batch can reach ~0.05x standard input price |
| Trade-off | asynchronous: results arrive within a window (up to 24h), not immediately |

- **Use for:** overnight report generation, backfilling analyses across historical periods, bulk classification of data-quality issues.
- **Never for:** interactive user queries.

Notebook flow:
1. Economics (multipliers)
2. Workload: 200 data-quality classification tasks with a long shared rubric
3. Baseline: synchronous calls via LangChain (and why `llm.batch()` is *not* the Batch API)
4. Submit one batch job of 200 requests (OpenAI Batch API)
5. Collect results, measure accuracy, compute real cost savings
6. Optional: Anthropic Message Batches with explicit `cache_control`
7. Decision rules and production checklist

In [ ]:
# %pip install -q "langchain-openai==1.2.2" "langchain-core" pandas tiktoken

In [1]:
# Check package version

import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community",  "langchain-text-splitters",
            "langchain-openai", "openai", "langgraph","tiktoken" ]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


# langchain version: 1.3.6
# langchain-community version: 0.4.2
# langchain-text-splitters version: 1.1.2
# langchain-openai version: 1.2.2
# openai version: 2.54.0
# langgraph version: 1.2.4
# tiktoken version: 0.14.0

langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-text-splitters version: 1.1.2
langchain-openai version: 1.2.2
openai version: 2.54.0
langgraph version: 1.2.4
tiktoken version: 0.14.0


In [4]:
import os, json, time, random, getpass, datetime as dt
from pathlib import Path

import pandas as pd
import tiktoken
import langchain, langchain_openai
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from openai import OpenAI

In [5]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

MODEL = "gpt-4.1-mini"   # any model that supports the Batch API; change if needed
client = OpenAI()
WORKDIR = Path("output_batch_demo"); WORKDIR.mkdir(exist_ok=True)

OPENAI_API_KEY:  ········


## 1. The economics

Price multipliers relative to the standard input price (1.0x):

- **Cache read** is cheaper than normal input. The multiplier is provider/model specific (Anthropic: 0.1x; OpenAI: model-dependent, automatic for prompts >= 1024 tokens).
- **Batch** halves everything.
- Discounts **multiply**: 0.1 x 0.5 = 0.05x for cached input in a batch.

In [6]:
BATCH = 0.5
CACHE_READ = 0.1   # Anthropic-style. Check your provider's pricing page.

econ = pd.DataFrame(
    [
        ("Standard, uncached", 1.0, 1.0),
        ("Standard, cache read", CACHE_READ, 1.0),
        ("Batch, uncached", BATCH, BATCH),
        ("Batch, cache read", CACHE_READ * BATCH, BATCH),
    ],
    columns=["mode", "input_multiplier", "output_multiplier"],
)
econ

,mode,input_multiplier,output_multiplier
0,"Standard, uncached",1.00,1.0
1,"Standard, cache read",0.10,1.0
2,"Batch, uncached",0.50,0.5
3,"Batch, cache read",0.05,0.5


## 2. The workload: bulk data-quality classification

Agent task: For each row of an orders table, label the data-quality issue. Every request shares one long **static rubric** (system prompt) and differs only in the record. That shared prefix is what caching targets; the 200 independent requests are what batching targets.

- **Rule for caching**: static content first, variable content last.

In [7]:
random.seed(7)
LABELS = ["OK", "MISSING_EMAIL", "INVALID_DATE", "NEGATIVE_AMOUNT", "BAD_COUNTRY_CODE", "OUTLIER_AMOUNT"]
COUNTRIES = ["US", "IN", "DE", "GB", "FR", "BR"]


def make_record(i):
    rec = {
        "order_id": f"ORD-{100000 + i}",
        "customer_email": f"user{i}@example.com",
        "order_date": (dt.date(2024, 1, 1) + dt.timedelta(days=random.randint(0, 900))).isoformat(),
        "amount": round(random.uniform(5, 1500), 2),
        "quantity": random.randint(1, 10),
        "country": random.choice(COUNTRIES),
    }
    label = random.choices(LABELS, weights=[55, 9, 9, 9, 9, 9])[0]
    
    if label == "MISSING_EMAIL":
        rec["customer_email"] = random.choice([None, "", "   "])
    elif label == "INVALID_DATE":
        rec["order_date"] = random.choice(["2024-02-30", "13/45/2024", "yesterday", "10-18-2026"])
    elif label == "NEGATIVE_AMOUNT":
        rec["amount"] = -abs(rec["amount"])
    elif label == "BAD_COUNTRY_CODE":
        rec["country"] = random.choice(["usa", "us", "United States", "U.S."])
    elif label == "OUTLIER_AMOUNT":
        rec["amount"] = round(random.uniform(80000, 500000), 2)
    return rec, label


N = 200
records, truth = zip(*(make_record(i) for i in range(N)))
truth_by_id = {r["order_id"]: t for r, t in zip(records, truth)}

pd.DataFrame(records).head()

,order_id,customer_email,order_date,amount,quantity,country
0,ORD-100000,user0@example.com,2024-11-27,1422.06,7,BR
1,ORD-100001,user1@example.com,2026-04-20,806.14,6,FR
2,ORD-100002,user2@example.com,2025-06-03,325.97,2,GB
3,ORD-100003,user3@example.com,2024-09-03,140.62,7,usa
4,ORD-100004,,2024-08-16,947.79,10,US


In [8]:
SYSTEM_PROMPT = """You are a data quality auditor for an e-commerce orders table.
For each record you receive, assign exactly ONE label from the list below and explain briefly.
Respond with a single JSON object and nothing else.

## Table schema
- order_id (string): unique, format "ORD-" followed by 6 digits.
- customer_email (string): must contain exactly one "@" and a domain containing at least one ".".
- order_date (string): must be an ISO 8601 date, YYYY-MM-DD, a real calendar date, between 2020-01-01 and 2026-12-31.
- amount (number): order total in USD. Must be greater than 0. Typical range is 5 to 2000.
- quantity (integer): must be 1 or greater.
- country (string): ISO 3166-1 alpha-2 code in UPPERCASE, for example "US", "IN", "DE", "GB", "FR", "BR".

## Labels
OK
  All fields satisfy the schema above. No issue found.
MISSING_EMAIL
  customer_email is null, an empty string, or contains only whitespace.
INVALID_DATE
  order_date is not in YYYY-MM-DD format, is not a real calendar date, or is outside the allowed range.
  Examples: "2024-02-30" (no such day), "13/45/2024" (wrong format), "yesterday" (not a date), "10-18-2026" (wrong order of parts).
NEGATIVE_AMOUNT
  amount is zero or negative. Refunds are stored in a separate table, so a negative amount here is always an error.
BAD_COUNTRY_CODE
  country is not exactly two uppercase letters.
  Examples: "usa" (three letters), "us" (lowercase), "United States" (full name), "U.S." (punctuation).
OUTLIER_AMOUNT
  amount is greater than 50000. Such values are almost always currency or decimal-point errors and must be reviewed manually.

## Precedence
If a record has more than one issue, report the first applicable label in this order:
MISSING_EMAIL, INVALID_DATE, NEGATIVE_AMOUNT, BAD_COUNTRY_CODE, OUTLIER_AMOUNT.

## Output format
{"label": "<one of the labels>", "field": "<name of the offending field, or null for OK>", "reason": "<at most 15 words>"}
Do not add other keys. Do not wrap the JSON in markdown. Do not add commentary.

## Worked examples
Record: {"order_id": "ORD-000001", "customer_email": "a@b.com", "order_date": "2025-03-14", "amount": 120.5, "quantity": 2, "country": "US"}
Output: {"label": "OK", "field": null, "reason": "All fields valid."}

Record: {"order_id": "ORD-000002", "customer_email": null, "order_date": "2025-03-14", "amount": 80.0, "quantity": 1, "country": "IN"}
Output: {"label": "MISSING_EMAIL", "field": "customer_email", "reason": "Email is null."}

Record: {"order_id": "ORD-000003", "customer_email": "c@d.org", "order_date": "2025-02-30", "amount": 45.0, "quantity": 3, "country": "DE"}
Output: {"label": "INVALID_DATE", "field": "order_date", "reason": "February 30 does not exist."}

Record: {"order_id": "ORD-000004", "customer_email": "e@f.net", "order_date": "2025-06-01", "amount": -19.99, "quantity": 1, "country": "GB"}
Output: {"label": "NEGATIVE_AMOUNT", "field": "amount", "reason": "Amount is negative."}

Record: {"order_id": "ORD-000005", "customer_email": "g@h.io", "order_date": "2025-07-21", "amount": 310.0, "quantity": 4, "country": "usa"}
Output: {"label": "BAD_COUNTRY_CODE", "field": "country", "reason": "Country must be two uppercase letters."}

Record: {"order_id": "ORD-000006", "customer_email": "i@j.com", "order_date": "2025-09-09", "amount": 250000.0, "quantity": 1, "country": "FR"}
Output: {"label": "OUTLIER_AMOUNT", "field": "amount", "reason": "Amount exceeds 50000."}

Record: {"order_id": "ORD-000007", "customer_email": "   ", "order_date": "13/45/2024", "amount": 60.0, "quantity": 1, "country": "BR"}
Output: {"label": "MISSING_EMAIL", "field": "customer_email", "reason": "Email is whitespace; precedence applies."}

Record: {"order_id": "ORD-000008", "customer_email": "k@l.com", "order_date": "2024-12-31", "amount": 50000.0, "quantity": 1, "country": "US"}
Output: {"label": "OK", "field": null, "reason": "Exactly 50000 is not above the threshold."}

## Edge cases
- Quantity problems are out of scope for this audit; ignore quantity unless instructed otherwise.
- Treat the strings "null", "None" and "N/A" in customer_email as missing.
- Dates with a valid format but in the future beyond 2026-12-31 are INVALID_DATE.
- Never invent fields that are not present in the record.
- Treat amounts supplied as strings (for example "120.50") as numbers when they parse cleanly; otherwise report the amount field.
- Whitespace around otherwise valid values (for example " US " or " 2025-01-01 ") does not change the label unless the rule above says so.
- A date such as 2020-01-01 or 2026-12-31 is inside the allowed range; 2019-12-31 and 2027-01-01 are outside it.
- Country codes must match the list of ISO codes exactly; "UK" is not valid because the ISO code for the United Kingdom is "GB".
- Keep the reason short, factual and specific to the offending value; do not repeat the whole record.
- If the record is not valid JSON, respond with {"label": "INVALID_DATE", "field": "order_date", "reason": "Unparseable record."}.
"""

enc = tiktoken.get_encoding("o200k_base")
SYSTEM_TOKENS = len(enc.encode(SYSTEM_PROMPT))
print("system prompt tokens:", SYSTEM_TOKENS)
if SYSTEM_TOKENS < 1024:
    print("WARNING: below the 1024-token minimum for automatic prompt caching; extend the rubric.")


def user_message(rec):
    return "Classify this record:\n" + json.dumps(rec)

system prompt tokens: 1444


## 3. Baseline: synchronous calls

This is the interactive path: one blocking request per record, full price. We run 5 to see latency and whether automatic caching kicks in (`cache_read` in `usage_metadata`).

In [9]:
llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}},
)

sync_usage = []
t0 = time.time()
for rec in records[:5]:
    msg = llm.invoke([SystemMessage(SYSTEM_PROMPT), HumanMessage(user_message(rec))])
    um = msg.usage_metadata
    cached = (um.get("input_token_details") or {}).get("cache_read", 0)
    sync_usage.append({"input": um["input_tokens"], "cached": cached, "output": um["output_tokens"]})
    print(rec["order_id"], msg.content, "| cache_read:", cached)

print(f"\n5 sync calls: {time.time() - t0:.1f}s -> ~{(time.time() - t0) / 5 * N / 60:.1f} min for {N} sequential calls")
pd.DataFrame(sync_usage)

ORD-100000 {"label": "OK", "field": null, "reason": "All fields valid."} | cache_read: 0
ORD-100001 {"label": "OK", "field": null, "reason": "All fields valid."} | cache_read: 1280
ORD-100002 {"label": "OK", "field": null, "reason": "All fields valid."} | cache_read: 1280
ORD-100003 {"label": "BAD_COUNTRY_CODE", "field": "country", "reason": "Country must be two uppercase letters."} | cache_read: 1280
ORD-100004 {"label": "MISSING_EMAIL", "field": "customer_email", "reason": "Email is empty string."} | cache_read: 1280

5 sync calls: 6.3s -> ~4.2 min for 200 sequential calls


,input,cached,output
0,1513,0,20
1,1512,1280,20
2,1512,1280,20
3,1512,1280,27
4,1507,1280,25


#### Observation: 
- First call has no cache, but subsequent calls are read through cache memory

**Gotcha:** `llm.batch([...])` in LangChain runs requests *concurrently* through the normal API. It improves wall-clock time but gives **no discount**. The 50% discount requires the provider's asynchronous Batch API, used directly below.

## 4. Submit one batch job for all 200 requests

### Steps: 
- write a JSONL file (one request per line, each with a unique `custom_id`) -> upload -> create batch. 

Results come back **unordered**, so `custom_id` is how you map outputs to inputs.

Limits (OpenAI): up to 50,000 requests and 200 MB per file; completion window `24h`.

In [10]:
def build_request(rec):
    return {
        "custom_id": f"dq-{rec['order_id']}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": MODEL,
            "temperature": 0,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},   # static prefix first
                {"role": "user", "content": user_message(rec)},  # variable part last
            ],
        },
    }


input_path = WORKDIR / "batch_input.jsonl"
with input_path.open("w") as f:
    for rec in records:
        f.write(json.dumps(build_request(rec)) + "\n")

print(f"{N} requests, {input_path.stat().st_size / 1024:.0f} KB")

200 requests, 1136 KB


In [11]:
batch_id_path = WORKDIR / "batch_id.txt"

if batch_id_path.exists():
    BATCH_ID = batch_id_path.read_text().strip()
    print("Resuming existing batch:", BATCH_ID)
else:
    with input_path.open("rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"job": "dq-classification-demo"},
    )
    BATCH_ID = batch.id
    batch_id_path.write_text(BATCH_ID)
    print("Submitted batch:", BATCH_ID, "| status:", batch.status)

Submitted batch: batch_6ad4cdc05e488190ae0aff9a1b04fe4c | status: validating


The job is now running server-side. In production you would stop here, persist `BATCH_ID`, and let a scheduler collect results later. The cell below polls so the demo completes in one sitting; if it times out, re-run it later (the ID is saved to disk).

In [12]:
# Following took 5 minutes

TERMINAL = {"completed", "failed", "expired", "cancelled"}


def wait_for_batch(batch_id, poll_seconds=30, timeout_seconds=1800):
    start = time.time()
    while True:
        b = client.batches.retrieve(batch_id)
        c = b.request_counts
        print(f"{b.status:12s} done={c.completed}/{c.total} failed={c.failed}")
        if b.status in TERMINAL or time.time() - start > timeout_seconds:
            return b
        time.sleep(poll_seconds)


batch = wait_for_batch(BATCH_ID)

validating   done=0/0 failed=0
validating   done=0/0 failed=0
in_progress  done=0/200 failed=0
in_progress  done=16/200 failed=0
in_progress  done=155/200 failed=0
completed    done=200/200 failed=0


## 5. Collect results, check quality, compute real savings

In [13]:
if batch.status != "completed":
    raise RuntimeError(f"Batch not finished (status={batch.status}). Re-run the polling cell later.")

out_text = client.files.content(batch.output_file_id).text
if batch.error_file_id:
    print("Errors:\n", client.files.content(batch.error_file_id).text[:1000])

rows = []
for line in out_text.splitlines():
    r = json.loads(line)
    order_id = r["custom_id"].removeprefix("dq-")
    resp = r.get("response")
    if not resp or resp["status_code"] != 200:
        rows.append({"order_id": order_id, "error": json.dumps(r.get("error") or resp)})
        continue
    body = resp["body"]
    u = body["usage"]
    pred = json.loads(body["choices"][0]["message"]["content"])
    rows.append({
        "order_id": order_id,
        "pred": pred.get("label"),
        "reason": pred.get("reason"),
        "input": u["prompt_tokens"],
        "cached": (u.get("prompt_tokens_details") or {}).get("cached_tokens", 0),
        "output": u["completion_tokens"],
    })

res = pd.DataFrame(rows)
res["truth"] = res["order_id"].map(truth_by_id)
ok = res[res["pred"].notna()]
print(f"succeeded: {len(ok)}/{N} | accuracy vs ground truth: {(ok['pred'] == ok['truth']).mean():.1%}")
res.head()

succeeded: 200/200 | accuracy vs ground truth: 100.0%


,order_id,pred,reason,input,cached,output,truth
0,ORD-100000,OK,All fields valid.,1513,0,20,OK
1,ORD-100001,OK,All fields valid.,1512,0,20,OK
2,ORD-100002,OK,All fields valid.,1512,0,20,OK
3,ORD-100003,BAD_COUNTRY_CODE,Country must be two uppercase letters.,1512,0,27,BAD_COUNTRY_CODE
4,ORD-100004,MISSING_EMAIL,Email is empty string.,1507,0,25,MISSING_EMAIL


In [14]:
# Retry pattern: collect failed/expired items and resubmit them as a new (smaller) batch.

failed_ids = sorted(set(truth_by_id) - set(ok["order_id"]))

print("to retry:", len(failed_ids))

to retry: 0


### Cost comparison

Prices are **USD per 1M tokens** and are placeholders for `gpt-4.1-mini` — verify against the current pricing page and edit.

Four scenarios, all using the token counts actually observed in the batch:
1. Standard, no caching
2. Standard, with the cache hits observed
3. Batch, with the cache hits observed (what you actually paid)
4. Batch, ideal caching (every request except the first reads the full rubric from cache)

Cache hits inside a batch are **best-effort** because requests are processed asynchronously and not strictly in order.

In [15]:
PRICE = {"input": 0.40, "cached_input": 0.10, "output": 1.60}   # per 1M tokens; VERIFY


def cost(input_tok, cached_tok, output_tok, batch=False, use_cache=True):
    cached = cached_tok if use_cache else 0
    usd = ((input_tok - cached) * PRICE["input"]
           + cached * PRICE["cached_input"]
           + output_tok * PRICE["output"]) / 1e6
    return usd * (0.5 if batch else 1.0)


tot_in, tot_cached, tot_out = ok["input"].sum(), ok["cached"].sum(), ok["output"].sum()
ideal_cached = min((len(ok) - 1) * SYSTEM_TOKENS, tot_in)

scenarios = {
    "1. Standard, no cache": cost(tot_in, 0, tot_out, batch=False, use_cache=False),
    "2. Standard, observed cache": cost(tot_in, tot_cached, tot_out, batch=False),
    "3. Batch, observed cache (actual)": cost(tot_in, tot_cached, tot_out, batch=True),
    "4. Batch, ideal cache": cost(tot_in, ideal_cached, tot_out, batch=True),
}
base = scenarios["1. Standard, no cache"]
cmp_df = pd.DataFrame(
    {"cost_usd": scenarios.values(), "vs_baseline": [f"{v / base:.2f}x" for v in scenarios.values()],
     "savings": [f"{1 - v / base:.0%}" for v in scenarios.values()]},
    index=scenarios.keys(),
)
print(f"tokens: input={tot_in:,} (cached={tot_cached:,}) output={tot_out:,}")
cmp_df

tokens: input=302,411 (cached=0) output=4,472


,cost_usd,vs_baseline,savings
"1. Standard, no cache",0.128120,1.00x,0%
"2. Standard, observed cache",0.128120,1.00x,0%
"3. Batch, observed cache (actual)",0.064060,0.50x,50%
"4. Batch, ideal cache",0.020956,0.16x,84%


In [16]:
# Project to a production-size job using the observed per-request averages.
def project(n_requests):
    n = len(ok)
    i, c, o = tot_in / n * n_requests, tot_cached / n * n_requests, tot_out / n * n_requests
    return pd.Series({
        "standard_no_cache": cost(i, 0, o, False, False),
        "batch_with_cache": cost(i, c, o, True),
    })


pd.DataFrame({n: project(n) for n in [200, 10_000, 50_000]}).T.assign(
    saved=lambda d: 1 - d["batch_with_cache"] / d["standard_no_cache"]
)

,standard_no_cache,batch_with_cache,saved
200,0.12812,0.06406,0.5
10000,6.40598,3.20299,0.5
50000,32.02990,16.01495,0.5


# STOP

## 6. Optional: Anthropic Message Batches with explicit caching

Anthropic requires you to mark the cacheable prefix with `cache_control`. Cache read is 0.1x input and batch is 0.5x, so cached input in a batch costs ~0.05x. Because a batch can take longer than the default 5-minute cache lifetime, use the **1-hour TTL** so hits survive across the job. Cache hits remain best-effort within batches.

Set `RUN_ANTHROPIC = True` and provide `ANTHROPIC_API_KEY` to run. Minimum cacheable prefix length varies by model; extend the rubric if `cache_creation_input_tokens` and `cache_read_input_tokens` both come back 0.

In [ ]:
RUN_ANTHROPIC = False
A_MODEL = "claude-sonnet-5-5"

if RUN_ANTHROPIC:
    %pip install -q anthropic
    import anthropic
    if not os.getenv("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
    a_client = anthropic.Anthropic()

    a_requests = [
        {
            "custom_id": f"dq-{rec['order_id']}",
            "params": {
                "model": A_MODEL,
                "max_tokens": 200,
                "temperature": 0,
                "system": [{
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                    "cache_control": {"type": "ephemeral", "ttl": "1h"},
                }],
                "messages": [{"role": "user", "content": user_message(rec)}],
            },
        }
        for rec in records
    ]

    a_batch = a_client.messages.batches.create(requests=a_requests)
    print("submitted:", a_batch.id)

    while True:
        a_batch = a_client.messages.batches.retrieve(a_batch.id)
        print(a_batch.processing_status, a_batch.request_counts)
        if a_batch.processing_status == "ended":
            break
        time.sleep(30)

    a_rows = []
    for item in a_client.messages.batches.results(a_batch.id):
        if item.result.type != "succeeded":
            a_rows.append({"custom_id": item.custom_id, "status": item.result.type})
            continue
        u = item.result.message.usage
        a_rows.append({
            "custom_id": item.custom_id, "status": "succeeded",
            "input": u.input_tokens, "cache_write": u.cache_creation_input_tokens,
            "cache_read": u.cache_read_input_tokens, "output": u.output_tokens,
        })
    a_df = pd.DataFrame(a_rows)
    display(a_df.describe(include="all"))

## 7. When to batch

| Workload | Batch? | Why |
|---|---|---|
| Overnight report generation | Yes | Deadline is hours away |
| Backfill analyses across historical periods | Yes | Thousands of independent requests |
| Bulk classification of data-quality issues | Yes | Same prompt prefix, many records |
| Interactive user query | **No** | User is waiting; latency matters |
| Multi-step agent loop where step N depends on step N-1 | **No** (per step) | Each step would wait up to the full window |

Batch the *independent fan-out* parts of an agent pipeline, not the sequential reasoning loop.

In [ ]:
def should_batch(interactive: bool, deadline_hours: float, n_requests: int, independent: bool = True) -> bool:
    if interactive or not independent:
        return False
    return deadline_hours >= 24 and n_requests >= 1   # window is up to 24h; many batches finish far sooner


for case in [
    dict(interactive=True,  deadline_hours=0.01, n_requests=1),
    dict(interactive=False, deadline_hours=48,   n_requests=5000),
    dict(interactive=False, deadline_hours=2,    n_requests=5000),
]:
    print(case, "->", should_batch(**case))

### Production checklist
- Unique, deterministic `custom_id` (e.g. `{job}-{period}-{row_id}`) so retries are idempotent.
- Persist the batch ID before doing anything else; collect results from a separate scheduled job.
- Results are unordered; always join by `custom_id`.
- Resubmit failed/expired items as a new batch; check the error file.
- Keep static content first, variable content last in every request.
- Split jobs that exceed provider limits (OpenAI: 50,000 requests / 200 MB per batch file; Anthropic: 100,000 requests / 256 MB).
- Batch quotas are separate from synchronous rate limits; check them for large backfills.
- Validate output on a sample (accuracy table above) before scaling to the full backfill.

### Exercises
1. Change the rubric so it is under 1024 tokens and compare `cached` in the results.
2. Put the record *before* the rubric in the messages and observe the cache hits drop.
3. Resubmit `failed_ids` as a second batch and merge results.
4. Run section 6 and compare `cache_read` counts with the OpenAI run.